# ⚖️ Nirnay — typed decisions on CPU

Ask **choice**, **yes / no** and **rating** questions about any text or JSON and get calibrated probabilities in one
pass: no GPU, no text generation, and option order can never change the answer. Same question format as Laya / Jev
(`/v1/systemone`).

[Model card & benchmarks](https://huggingface.co/nandukmelath/nirnay) · [Code](https://github.com/nandukmelath/nirnay)

This notebook runs on the free Colab **CPU** runtime; no GPU is needed.

In [ ]:
!pip install -q git+https://github.com/nandukmelath/nirnay

In [ ]:
import nirnay

m = nirnay.load("nandukmelath/nirnay")                       # v2 (bge-base): most accurate
# m = nirnay.load("nandukmelath/nirnay", subfolder="fast")   # v1 (bge-small): about 3x faster

## One text, several typed questions, one call

`choice` picks one of N labels, `noul` answers yes / no with P(yes), `score` rates on ordered levels.

In [ ]:
questions = {
    "team":     {"type": "choice", "instructions": "Which team should handle this?",
                 "criteria": {"billing": "payments and refunds", "technical": "bugs and integrations",
                              "sales": "pricing and plans"}},
    "urgent":   {"type": "noul", "instructions": "Is this ticket urgent?"},
    "severity": {"type": "score", "instructions": "How severe is the problem?",
                 "criteria": ["minor", "moderate", "severe"]},
}
ticket = {"ticket": {"subject": "Payout failing", "body": "My payouts have failed for 3 days, please help ASAP"}}

r = m.predict(ticket, questions)
a = r["answers"]
print("team     :", a["team"]["choice"], a["team"]["probabilities"])
print("urgent   : P(yes) =", a["urgent"]["noul"])
print("severity : most likely %r, expected level %.2f on 0-2" % (a["severity"]["legend"][a["severity"]["choice"]], a["severity"]["score"]))
print("latency  :", r["latency_ms"], "ms")

## Option order can't change the answer

Every option is encoded on its own, so shuffling the labels gives the same probability per label.

In [ ]:
text = "The new firmware made the drone drift left during hover; calibration did not help."
labels = ["world news", "sports", "business", "science and technology"]
for order in (labels, labels[::-1]):
    q = {"topic": {"type": "choice", "instructions": "What is this text about?", "criteria": order}}
    print(m.predict(text, q)["answers"]["topic"]["probabilities"])

## Ten questions cost little more than one

The text is read once per call and shared by every question.

In [ ]:
import time

subjects = ["money", "a deadline", "a refund", "a software bug", "a person's name",
            "a date", "a product", "anger", "praise", "a question"]
many = {"q%d" % i: {"type": "noul", "instructions": "Does the text mention %s?" % s} for i, s in enumerate(subjects)}
for qs in ({"q0": many["q0"]}, many):
    m.predict(ticket, qs)                                    # first call caches the question encodings
    t = time.perf_counter()
    m.predict(ticket, qs)
    print("%2d question(s): %.0f ms" % (len(qs), (time.perf_counter() - t) * 1000))

## Teach it with a few examples, no training: `learn()`

Labelled examples are attached to the question and used from the next call on.

In [ ]:
team_q = questions["team"]
examples = [
    ("I was charged twice this month", "billing"),
    ("My refund still hasn't arrived after 10 days", "billing"),
    ("The API returns 500 errors since this morning", "technical"),
    ("The app crashes when I upload a photo", "technical"),
    ("Do you offer discounts for nonprofits?", "sales"),
    ("What does the enterprise plan include?", "sales"),
]
before = m.predict("Can I get a quote for 50 seats?", {"team": team_q})["answers"]["team"]
m.learn("team", team_q, examples)
after = m.predict("Can I get a quote for 50 seats?", {"team": team_q})["answers"]["team"]
print("before:", before["choice"], before["probabilities"])
print("after :", after["choice"], after["probabilities"])

## Many texts, same questions: `predict_batch()`

In [ ]:
reviews = ["Absolutely love it, works perfectly and arrived early.",
           "It's okay. Does the job, nothing special.",
           "Broke after two days and support never answered."]
rating = {"stars": {"type": "score", "instructions": "How satisfied is the customer?",
                    "criteria": ["very unhappy", "unhappy", "neutral", "happy", "very happy"]}}
for text, r in zip(reviews, m.predict_batch(reviews, rating)):
    s = r["answers"]["stars"]
    print("%.2f  %-13s %s" % (s["score"], s["legend"][s["choice"]], text))

## Use it from your own code

* HTTP server with the Laya / Jev wire format: `python -m nirnay.server --port 8000` → `POST /v1/systemone`
* MCP server for local agents: `python -m nirnay.mcp_server`
* Command line: `nirnay "text" --choice "billing,technical,sales" --ask "Which team?"`

Benchmarks against Laya and Jev, limitations and training details:
[model card](https://huggingface.co/nandukmelath/nirnay) · [GitHub](https://github.com/nandukmelath/nirnay)